# 大模型 LoRA 微调最佳实践 —— ms-swift + Qwen3-4B

### 平台操作步骤（在打开本 Notebook 之前）

1. **登录 AI Studio 控制台** → 「模型训练」→「Notebook 建模」→「创建实例」
2. **选择资源规格**：PPU 1 卡，CPU 8 核 + 内存 64GB
3. **选择镜像**：公共镜像（PPU 适配版，ms-swift 预装，ppu1.4.2 系列）
4. **挂载模型**：添加模型挂载 → 选择 Qwen3-4B → 挂载路径 `/model`
5. **挂载存储**：添加持久化存储 → 挂载路径 `/mnt/workspace`（用于保存训练产出，实例重启不丢失）
6. 点击「确认」创建，等待状态变为「运行中」后打开 Jupyter

## 0. 路径配置

根据创建 Notebook 实例时的实际挂载路径修改以下变量。

In [ ]:
# ============================================================
# 路径配置 - 根据实际挂载路径修改
# ============================================================

# 模型路径：Qwen3-4B 权重挂载到容器的路径（目录下直接是权重文件）
MODEL_PATH = '/model'

# 训练输出目录：挂载的持久化存储路径（实例重启不丢失）
OUTPUT_DIR = '/mnt/workspace/output/qwen3-4b-lora'

# ============================================================
import os

assert os.path.isdir(MODEL_PATH), f'模型路径不存在: {MODEL_PATH}\n请检查实例创建时是否正确配置了模型挂载'
print(f'[OK] 模型路径: {MODEL_PATH}')
print(f'     模型文件: {os.listdir(MODEL_PATH)[:8]}')

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f'[OK] 输出目录: {OUTPUT_DIR}')

## 1. 环境检查

In [ ]:
import torch

print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'加速卡: {torch.cuda.get_device_name(0)}')

## 2. 数据准备

ms-swift 支持多种数据集格式。本实践使用 **Alpaca 格式**（instruction / input / output），这是一种常用的指令微调数据格式。

下面的代码会在当前目录下生成 `data/custom_train.json`，包含 15 条示例数据。实际使用时建议准备 **500 - 5000 条**高质量的领域数据。

In [ ]:
import json
import os

os.makedirs('data', exist_ok=True)

custom_data = [
    {
        "instruction": "你是谁？你有什么能力？",
        "output": "我是基于通义千问Qwen3大模型微调的AI助手，专注于AI模型开发与平台使用方面的问题解答。我可以帮助你：\n1. 解答模型训练、微调、部署相关的技术问题\n2. 指导AI Studio平台的使用方法\n3. 提供模型调优和性能优化的建议\n4. 解释机器学习和深度学习的核心概念"
    },
    {
        "instruction": "什么是LoRA微调？为什么它很重要？",
        "output": "LoRA（Low-Rank Adaptation，低秩适配）是一种参数高效的大模型微调方法。其核心原理是：冻结预训练模型的所有参数，在Transformer的注意力层旁路插入两个小的低秩矩阵（A和B），仅训练这两个矩阵。\n\n为什么重要：\n- 可训练参数量降低到原模型的0.1%~1%，大幅节省显存\n- 单张消费级GPU即可微调7B甚至13B模型\n- 训练速度快，通常几小时即可完成\n- 可以保存多个轻量级adapter，针对不同任务灵活切换\n- 微调效果接近全量微调"
    },
    {
        "instruction": "如何在AI Studio平台创建一个Notebook实例？",
        "output": "在AI Studio平台创建Notebook实例的步骤如下：\n\n1. 登录AI Studio控制台\n2. 在左侧菜单点击「模型训练」→「Notebook建模」\n3. 点击「创建实例」按钮\n4. 填写配置信息：\n   - 实例名称：输入有意义的名称\n   - 资源配额：选择已申请的资源配额\n   - 资源规格：根据任务选择CPU核数、内存和PPU数量\n   - 镜像选择：推荐使用PPU适配镜像\n   - 编辑环境：选择Jupyter\n5. 挂载模型权重到/model目录\n6. 设置登录账号和密码\n7. 点击「确认」创建\n8. 等待状态变为「运行中」后，点击「打开」进入Jupyter环境"
    },
    {
        "instruction": "SFT和RLHF有什么区别？",
        "output": "SFT（Supervised Fine-Tuning，监督微调）和RLHF（Reinforcement Learning from Human Feedback，基于人类反馈的强化学习）是大模型后训练的两个不同阶段：\n\nSFT：\n- 使用标注好的指令-回复对进行监督学习\n- 目标是让模型学会遵循指令的格式和风格\n- 数据格式简单，通常是{instruction, output}对\n- 训练过程类似传统的监督学习\n\nRLHF：\n- 在SFT之后进行，使用人类偏好数据进一步优化\n- 通过奖励模型（Reward Model）来评估回复质量\n- 使用PPO/DPO/GRPO等强化学习算法优化策略\n- 目标是让模型输出更符合人类偏好的回复\n\n通常的训练流程是：预训练 → SFT → RLHF"
    },
    {
        "instruction": "解释一下什么是QLoRA。",
        "output": "QLoRA（Quantized LoRA）是LoRA的进阶版本，在LoRA基础上增加了模型量化技术：\n\n核心思路：\n1. 将基座模型量化为4-bit精度（NF4格式）加载到显存\n2. 在量化模型上应用LoRA适配器\n3. LoRA部分以bf16/fp16精度进行训练\n4. 使用双重量化（Double Quantization）进一步压缩内存\n\n显存对比（以7B模型为例）：\n- 全量微调：约60GB显存\n- LoRA（bf16）：约18-20GB显存\n- QLoRA（4-bit）：约6-8GB显存\n\n这意味着QLoRA可以在单张T4（16GB）GPU上微调7B模型，极大降低了微调的硬件门槛。"
    },
    {
        "instruction": "模型训练时出现OOM（显存不足）怎么办？",
        "output": "OOM（Out of Memory）是模型训练中最常见的问题。以下是按优先级排列的解决方案：\n\n1. 减小batch_size：将per_device_train_batch_size降到1，同时增大gradient_accumulation_steps保持等效batch_size\n2. 减小序列长度：降低max_length，如从2048降到512\n3. 开启梯度检查点：设置gradient_checkpointing=true，用计算换显存\n4. 使用QLoRA：4-bit量化可减少约50%显存\n5. 减小LoRA rank：从16降到8或4\n6. 使用更小的模型：如从7B换到4B或1.5B\n\n如果以上方法都不行，需要申请更大显存的资源。"
    },
    {
        "instruction": "ms-swift框架有什么优势？",
        "output": "ms-swift是ModelScope社区提供的大模型微调部署框架，主要优势包括：\n\n1. 模型支持广：支持600+纯文本大模型和400+多模态模型\n2. 训练方法全：支持SFT/DPO/PPO/GRPO/KTO等多种后训练方式\n3. 命令行简洁：一行swift sft命令即可启动训练\n4. LoRA高效微调：内置LoRA/QLoRA/全量微调等多种方式\n5. 深度集成ModelScope生态：模型下载、数据集加载一站式\n6. 支持多模态：图片、音频、视频多模态理解与生成\n7. 集成评测框架EvalScope\n8. 支持分布式训练和部署"
    },
    {
        "instruction": "将以下文本翻译为英文。",
        "input": "人工智能正在深刻改变我们的生产和生活方式。",
        "output": "Artificial intelligence is profoundly changing our ways of production and daily life."
    },
    {
        "instruction": "请用简单的语言解释什么是Transformer架构。",
        "output": "Transformer是一种神经网络架构，是目前大模型的基础。简单来说：\n\n1. 自注意力机制：让模型能够同时关注输入序列中所有位置的信息，而不是像RNN那样逐个处理。就像阅读文章时能同时看到前后文。\n\n2. 编码器-解码器结构：编码器理解输入，解码器生成输出。GPT类模型只用解码器，BERT只用编码器。\n\n3. 并行计算：相比RNN的顺序处理，Transformer可以并行处理所有位置，训练速度大幅提升。\n\n4. 位置编码：由于并行处理丢失了顺序信息，通过位置编码告诉模型每个词的位置。\n\nTransformer的出现使得训练超大规模语言模型成为可能，催生了GPT、BERT、Qwen等一系列大模型。"
    },
    {
        "instruction": "Qwen3模型有什么特点？",
        "output": "Qwen3是阿里云通义千问团队推出的新一代大语言模型，主要特点包括：\n\n1. 思考模式（Thinking Mode）：模型可以在<think>标签内进行推理链思考，提升复杂问题的回答质量\n2. /no_think指令：在问题末尾加/no_think可跳过思考过程直接回答，适用于简单问答场景\n3. 多种参数规格：从0.6B到235B，覆盖不同资源需求\n4. 开源可商用：采用Apache 2.0协议开源\n5. 强大的多语言能力：支持中英日韩等多种语言\n6. 对齐优化：经过SFT和RLHF精心调优，回复质量高"
    },
    {
        "instruction": "什么是gradient_checkpointing？为什么要使用它？",
        "output": "梯度检查点（Gradient Checkpointing）是一种用计算时间换取显存空间的技术：\n\n原理：\n- 正常训练时，前向传播的所有中间激活值都保存在显存中，用于反向传播计算梯度\n- 梯度检查点只保存部分层的激活值（检查点），其余在反向传播时重新计算\n- 这样显存占用减少约30-50%，但训练时间增加约20-30%\n\n使用场景：\n- 显存不够时，优先开启（--gradient_checkpointing true）\n- 相比减小batch_size，梯度检查点对训练效果的影响更小\n- 几乎所有大模型训练框架都支持这个选项\n\n建议：显存充足时可以关闭以加速训练，显存紧张时务必开启。"
    },
    {
        "instruction": "如何评估微调后模型的效果？",
        "output": "评估微调后模型效果有以下几种方法：\n\n1. 人工评估（最直观）：\n   - 准备一组测试问题，对比微调前后的回答质量\n   - 关注回答的准确性、流畅性、格式遵循度\n\n2. 自动评测（推荐）：\n   - 使用EvalScope/OpenCompass等评测框架\n   - 在标准benchmark上评测（如C-Eval、MMLU）\n\n3. 训练指标监控：\n   - loss曲线：应持续下降并趋于平稳\n   - 验证集loss：如果上升说明过拟合\n\n4. A/B测试：\n   - 在实际业务场景中对比基座模型和微调模型\n   - 收集用户反馈\n\n注意：小数据集（<100条）微调容易过拟合，建议增加数据量或减少训练轮数。"
    },
    {
        "instruction": "什么是learning_rate warmup？为什么需要？",
        "output": "Learning rate warmup（学习率预热）是训练大模型时常用的学习率调度策略：\n\n原理：\n- 训练开始时不直接使用目标学习率，而是从一个很小的值逐步增加到目标值\n- 预热阶段通常占总训练步数的3-10%（warmup_ratio=0.05表示5%）\n- 预热后再按余弦退火等策略逐步降低\n\n为什么需要：\n1. 训练初期模型参数随机，梯度方差大，大学习率容易导致训练不稳定\n2. 预热让优化器的动量估计更准确（如Adam的一阶二阶矩估计）\n3. 防止训练初期loss剧烈震荡甚至发散\n4. 对大batch_size训练尤其重要\n\n实践建议：LoRA微调通常设置warmup_ratio=0.03~0.1"
    },
    {
        "instruction": "帮我写一段Python代码，实现简单的文本分类。",
        "output": "以下是一个使用sklearn实现文本分类的简单示例：\n\n```python\nfrom sklearn.feature_extraction.text import TfidfVectorizer\nfrom sklearn.naive_bayes import MultinomialNB\nfrom sklearn.model_selection import train_test_split\nfrom sklearn.metrics import classification_report\n\n# 示例数据\ntexts = [\n    '这个手机拍照效果很好', '电池续航太差了',\n    '屏幕显示非常清晰', '发货速度太慢',\n    '性价比很高推荐购买', '质量有问题退货了'\n]\nlabels = ['正面', '负面', '正面', '负面', '正面', '负面']\n\n# 特征提取\nvectorizer = TfidfVectorizer()\nX = vectorizer.fit_transform(texts)\n\n# 训练模型\nclf = MultinomialNB()\nclf.fit(X, labels)\n\n# 预测\ntest = vectorizer.transform(['这个产品非常好用'])\nprint(clf.predict(test))  # ['正面']\n```\n\n注意：实际项目中建议使用预训练语言模型（如BERT）做文本分类，效果更好。"
    },
    {
        "instruction": "模型微调完成后，如何部署上线？",
        "output": "模型微调完成后的部署流程：\n\n1. 模型导出：\n   - 使用swift export合并LoRA权重\n   - 导出完整模型到指定目录\n\n2. 部署方式选择：\n   - vLLM部署：高性能推理，支持连续批处理\n   - TGI部署：HuggingFace官方推理框架\n   - ModelScope部署：一键部署到ModelScope\n   - API服务：使用FastAPI封装推理接口\n\n3. AI Studio平台部署：\n   - 在控制台选择「模型部署」\n   - 选择导出的模型文件\n   - 配置资源规格和副本数\n   - 获取API endpoint进行调用\n\n4. 优化建议：\n   - 使用量化（INT8/INT4）减少推理资源需求\n   - 配置KV Cache优化推理速度\n   - 设置合理的max_tokens和并发数"
    },
]

# 写入训练数据文件
with open('data/custom_train.json', 'w', encoding='utf-8') as f:
    json.dump(custom_data, f, ensure_ascii=False, indent=2)

print(f'训练数据已生成到 data/custom_train.json，共 {len(custom_data)} 条')
print(f'\n数据格式预览（第 1 条）:')
print(json.dumps(custom_data[0], ensure_ascii=False, indent=2))

## 3. 执行 LoRA 微调

**Qwen3 特有参数说明：**
- `--model_type qwen3`：指定模型类型为 Qwen3
- `--loss_scale ignore_empty_think`：忽略空 `<think>` 标签的 loss，保留模型思考能力
- `--target_modules all-linear`：Qwen3 官方推荐的 LoRA 目标模块
- `--torch_dtype bfloat16`：PPU 支持 bf16 训练精度

**PPU 资源配置（8C64G 1 卡）下的推荐参数：**
- batch_size=2, max_length=2048, lora_rank=8 → 流畅运行
- 可适当增大 batch_size=4 或 lora_rank=16/32


In [ ]:
# 使用 ms-swift 进行 LoRA 微调
# 模型从本地挂载路径 /model 加载，无需联网下载
!swift sft \
    --model {MODEL_PATH} \
    --model_type qwen3 \
    --train_type lora \
    --dataset 'data/custom_train.json' \
    --torch_dtype bfloat16 \
    --lora_rank 8 \
    --lora_alpha 32 \
    --target_modules all-linear \
    --loss_scale ignore_empty_think \
    --num_train_epochs 5 \
    --per_device_train_batch_size 2 \
    --gradient_accumulation_steps 8 \
    --learning_rate 1e-4 \
    --max_length 2048 \
    --warmup_ratio 0.05 \
    --gradient_checkpointing true \
    --save_steps 20 \
    --logging_steps 1 \
    --output_dir {OUTPUT_DIR}

## 4. 微调效果验证

### 4.1 找到最新的 checkpoint

In [ ]:
import glob
import os

# ms-swift 3.9 输出目录结构: {OUTPUT_DIR}/{version_dir}/checkpoint-{step}
# 例如: v2-20260315-014206/checkpoint-5

# 方法 1：直接从训练日志中读取（最可靠）
args_path = None
for vdir in sorted(os.listdir(OUTPUT_DIR), reverse=True):
    candidate = os.path.join(OUTPUT_DIR, vdir, 'args.json')
    if os.path.isfile(candidate):
        args_path = candidate
        break

latest = None
if args_path:
    import json as _json
    with open(args_path) as f:
        args = _json.load(f)
    latest = args.get('last_model_checkpoint')

# 方法 2：glob 兜底
if not latest:
    checkpoint_dirs = sorted(glob.glob(f'{OUTPUT_DIR}/*/checkpoint-*'), key=os.path.getmtime)
    latest = checkpoint_dirs[-1] if checkpoint_dirs else None

if latest and os.path.isdir(latest):
    print(f'最新 checkpoint: {latest}')
    print(f'checkpoint 文件: {os.listdir(latest)[:10]}')
else:
    print(f'未找到有效 checkpoint')
    print(f'{OUTPUT_DIR} 目录内容：')
    for d in sorted(os.listdir(OUTPUT_DIR)):
        sub = os.path.join(OUTPUT_DIR, d)
        if os.path.isdir(sub):
            print(f'  {d}/ -> {os.listdir(sub)[:5]}')

### 4.2 批量验证

创建测试问题，使用 `swift infer` 进行批量推理验证微调效果。

In [ ]:
import json
import os

# 创建测试问题
test_questions = [
    {"instruction": "你是谁？"},
    {"instruction": "什么是LoRA微调？"},
    {"instruction": "模型训练时出现OOM怎么办？"},
]
with open('data/test_questions.json', 'w', encoding='utf-8') as f:
    json.dump(test_questions, f, ensure_ascii=False, indent=2)

# 使用 swift infer 批量推理
# --attn_impl eager: 使用纯 PyTorch attention 实现
assert latest, '未找到 checkpoint，请先运行上方训练步骤。'
infer_cmd = f'swift infer --model {MODEL_PATH} --model_type qwen3 --adapters {latest} --val_dataset data/test_questions.json --max_new_tokens 2048 --attn_impl eager'
print(f'执行命令:\n{infer_cmd}\n')
!{infer_cmd}

## 5. 模型导出

合并 LoRA 权重导出完整模型，保存到持久存储。

In [ ]:
EXPORT_DIR = '/mnt/workspace/output/qwen3-4b-merged-model'

assert latest, '未找到 checkpoint，请先运行训练步骤。'
export_cmd = f'swift export --adapters {latest} --merge_lora true --output_dir {EXPORT_DIR}'
print(f'执行命令:\n{export_cmd}\n')
!{export_cmd}

print(f'\n合并后的模型保存到: {EXPORT_DIR}')
for f in os.listdir(EXPORT_DIR):
    sz = os.path.getsize(f'{EXPORT_DIR}/{f}') / 1024**2
    print(f'  {f}: {sz:.1f} MB')